# AI623 PA2 — LLM Alignment: full pipeline (Kaggle)

Runs the whole coding section end-to-end: Reward Model → SFT → PPO → DPO → GRPO → RLVR → Evaluation.

**Before running:**
1. Settings → Accelerator → **GPU T4 x2** (or **GPU P100** if available).
2. Settings → Add-ons → Secrets → add `HF_TOKEN` (your Hugging Face token).
3. Settings → Internet → **On**.
4. Point `REPO_URL` in cell 1 at your GitHub repo.

Checkpoints are written to `/kaggle/working/pa2/` which Kaggle preserves as versioned notebook output.

**Expected total time on P100**: ~4–6 hours.

## 1 · Setup: clone repo, install deps, HF login

In [ ]:
import os, subprocess, sys, time

REPO_URL = "https://github.com/0x0shephard/assignment.git"   # <-- change if needed
REPO_DIR = "/kaggle/working/assignment"
CKPT_ROOT = "/kaggle/working/pa2"

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

if not os.path.isdir(REPO_DIR):
    subprocess.check_call(["git", "clone", REPO_URL, REPO_DIR])
else:
    subprocess.check_call(["git", "-C", REPO_DIR, "pull", "--rebase"])

os.makedirs(CKPT_ROOT, exist_ok=True)
os.chdir(REPO_DIR)
print("repo at:", REPO_DIR)
print("ckpts →:", CKPT_ROOT)
!nvidia-smi | head -14

In [ ]:
# Kaggle preinstalls most of what we need. bitsandbytes is Linux-only.
# torchao upgrade avoids the peft LoRA import error on newer torch.
!pip -q install -r requirements.txt bitsandbytes
!pip -q install -U "torchao>=0.16"

In [ ]:
# HF login via Kaggle secret. Optional — required only for gated models like
# meta-llama/Llama-3.2-1B-Instruct. SmolLM2 (our default) is public.
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = token
    from huggingface_hub import login
    login(token=token, add_to_git_credential=False)
    print("HF login OK")
except Exception as e:
    print(f"HF_TOKEN not found ({e.__class__.__name__}). Proceeding without login — fine while every model is public.")

## 2 · Config — one place to tune everything

SmolLM2-360M as both policy and RM/value backbone (proven throughput on T4). Change `BACKBONE` to Llama-3.2-1B-Instruct if you have access.

In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

POLICY     = "HuggingFaceTB/SmolLM2-360M"
BACKBONE   = "HuggingFaceTB/SmolLM2-360M"   # RM + value model

RM_DIR   = f"{CKPT_ROOT}/rm"
SFT_DIR  = f"{CKPT_ROOT}/sft"
PPO_DIR  = f"{CKPT_ROOT}/ppo"
DPO_DIR  = f"{CKPT_ROOT}/dpo"
GRPO_DIR = f"{CKPT_ROOT}/grpo"
RLVR_DIR = f"{CKPT_ROOT}/rlvr"
for d in [RM_DIR, SFT_DIR, PPO_DIR, DPO_DIR, GRPO_DIR, RLVR_DIR]:
    os.makedirs(d, exist_ok=True)
print("config OK")

## 2.5 · Kaggle-only runtime patch

Adds `torch.cuda.empty_cache()` at the top of each training loop to counter PyTorch 2.10's fragmented allocator on T4 GPUs. This is a Kaggle-specific memory tweak, not in the git repo (it's not a correctness fix).

In [ ]:
import ast
for name in ("train_ppo.py", "train_grpo.py", "train_rlvr.py", "train_dpo.py"):
    p = f"/kaggle/working/assignment/{name}"
    lines = open(p).readlines()
    if any("torch.cuda.empty_cache()" in ln for ln in lines):
        print(f"{name}: already patched"); continue
    for i, ln in enumerate(lines):
        s = ln.lstrip()
        if s.startswith("for step in range(1, args.steps + 1):") or s.startswith("for i, batch in enumerate(pbar):"):
            indent = ln[:len(ln) - len(ln.lstrip())] + "    "
            lines.insert(i + 1, f"{indent}if torch.cuda.is_available(): torch.cuda.empty_cache()\n")
            open(p, "w").writelines(lines)
            ast.parse(open(p).read())
            print(f"patched {name}"); break

## 3 · C0 sanity check (~1–2 min)
Confirms data pipeline, model loading, LoRA, and frozen-ref helper all work before we burn hours on training.

In [ ]:
!python verify_c0.py --limit 32 --policy $POLICY --backbone $BACKBONE

## 4 · Phase 3 — Reward Model (~15–30 min)
Target: **preference accuracy ≥ 0.60** on HH-RLHF test.

In [ ]:
!PYTORCH_ALLOC_CONF=expandable_segments:True python train_rm.py \
    --backbone $BACKBONE \
    --limit 4000 \
    --max_len 384 \
    --batch_size 8 \
    --lr 5e-4 \
    --log_every 25 \
    --out $RM_DIR
!cat $RM_DIR/summary.json

## 5 · Phase 4 — SFT warm-up (~30–40 min)
Trains π_θ^(0) on HH-RLHF chosen responses. This is the initialization for **every** RL method (PPO/DPO/GRPO/RLVR) and also serves as π_ref (KL anchor).

In [ ]:
!PYTORCH_ALLOC_CONF=expandable_segments:True python train_sft.py \
    --policy $POLICY \
    --limit 4000 \
    --max_len 384 \
    --batch_size 4 \
    --grad_accum 8 \
    --epochs 1 \
    --lr 2e-4 \
    --out $SFT_DIR
!cat $SFT_DIR/summary.json | python -m json.tool | head -40

## 6 · Phase 5 — PPO (~1–2 h)
Policy + reference (same base, adapters toggle) + separate Llama-family value backbone + frozen RM. 200 update steps × 4 mini-batch epochs.

**Watch for**: `rm` trending up, `kl_ref` bounded, `clip%` in 0.1–0.3 range.

In [ ]:
!PYTORCH_ALLOC_CONF=expandable_segments:True python train_ppo.py \
    --policy $POLICY \
    --backbone $BACKBONE \
    --sft_dir $SFT_DIR \
    --rm_dir  $RM_DIR \
    --steps 200 \
    --prompts_per_step 2 \
    --max_new_tokens 64 \
    --beta 0.1 \
    --eps_clip 0.2 \
    --lam 0.95 \
    --epochs_per_batch 2 \
    --policy_lr 1e-5 \
    --value_lr 1e-4 \
    --eval_every 25 \
    --out $PPO_DIR

## 7 · Phase 6 — DPO (~30–45 min)
Offline, no reward model. Init pref-acc should be ~0.50 (Δ_θ = Δ_ref at start); rising past 0.55 by step 100 confirms the reference model is truly frozen and masking is correct.

In [ ]:
!PYTORCH_ALLOC_CONF=expandable_segments:True python train_dpo.py \
    --policy $POLICY \
    --sft_dir $SFT_DIR \
    --limit 4000 \
    --max_len 384 \
    --batch_size 4 \
    --grad_accum 8 \
    --epochs 1 \
    --lr 5e-6 \
    --beta 0.1 \
    --eval_every 25 \
    --out $DPO_DIR
!cat $DPO_DIR/summary.json | python -m json.tool | head -40

## 8 · Phase 7 — GRPO (~1.5 h)
Critic-free. K=4 completions per prompt, group-relative advantage.

**Watch for**: `frac_degen` (all-K-same-reward batches) dropping over time.

In [ ]:
!PYTORCH_ALLOC_CONF=expandable_segments:True python train_grpo.py \
    --policy $POLICY \
    --backbone $BACKBONE \
    --sft_dir $SFT_DIR \
    --rm_dir  $RM_DIR \
    --steps 200 \
    --prompts_per_step 2 \
    --K 4 \
    --max_new_tokens 64 \
    --beta 0.1 \
    --eps_clip 0.2 \
    --kl_mode mc \
    --epochs_per_batch 1 \
    --lr 1e-5 \
    --eval_every 25 \
    --out $GRPO_DIR

## 9 · Phase 8 — RLVR on GSM8K (~1.5–2 h)
GRPO with a verifiable 0/1 reward. **Must init from plain SFT ckpt** (not from PPO/GRPO on HH-RLHF).

In [ ]:
!PYTORCH_ALLOC_CONF=expandable_segments:True python train_rlvr.py \
    --policy $POLICY \
    --sft_dir $SFT_DIR \
    --steps 300 \
    --K 4 \
    --prompts_per_step 2 \
    --max_new_tokens 128 \
    --beta 0.05 \
    --eps_clip 0.2 \
    --kl_mode mc \
    --lr 1e-5 \
    --eval_every 25 \
    --eval_n 200 \
    --out $RLVR_DIR

## 10 · Phase 10 — Evaluation (~10–15 min)
Greedy generations on 200 held-out HH-RLHF prompts. Reports RM win-rate vs SFT, KL from ref, sample + resource tables.

In [ ]:
!PYTORCH_ALLOC_CONF=expandable_segments:True python eval.py \
    --policy $POLICY \
    --backbone $BACKBONE \
    --sft_dir $SFT_DIR \
    --rm_dir  $RM_DIR \
    --aligned ppo=$PPO_DIR dpo=$DPO_DIR grpo=$GRPO_DIR \
    --n_prompts 200 \
    --max_new_tokens 96 \
    --batch_size 4 \
    --out $CKPT_ROOT/eval_results

In [ ]:
# Pretty-print the summary table
import json, pathlib
res = json.loads(pathlib.Path(f"{CKPT_ROOT}/eval_results/results.json").read_text())
print("n_prompts:", res["n_prompts"])
print("\n=== resource_table ===")
for r in res["resource_table"]:
    print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items()})
print("\n=== sample_table (first 2 prompts) ===")
for row in res["sample_table"][:2]:
    print("\n--- prompt tail:", row["prompt"][-140:])
    for m in res["resource_table"]:
        name = m["method"]
        cell = row[name]
        print(f"  [{name:>5}] rm={cell['rm_score']:+.3f}  {cell['response'][:200]}")

## 11 · (Optional) Phase 9 — one ablation for C7

Cheapest illustrative sweep: **DPO β**. Wires up Problem 3.2(a)'s regularization discussion.

In [ ]:
# Skip unless you have time budget — ~2 h.
!python ablations/sweep.py dpo_beta --out $CKPT_ROOT/runs -- \
    --policy $POLICY --sft_dir $SFT_DIR --limit 4096 \
    --max_len $MAX_LEN --batch_size 8 --grad_accum 4
!python ablations/sweep.py summarize --sweep_dirs $CKPT_ROOT/runs/dpo_beta/*

## 12 · Save Kaggle output as a versioned dataset

Everything under `/kaggle/working/` is preserved when you click **Save Version** (top right). You can attach the resulting dataset as input to a follow-up notebook to skip re-training.

Deliverables to include in your working document / viva:
- `pa2/rm/summary.json`             — RM preference accuracy
- `pa2/sft/summary.json`            — SFT PPL + sample generations
- `pa2/{ppo,grpo,rlvr}/log.json`    — per-step training curves
- `pa2/dpo/summary.json`
- `pa2/eval_results/results.json`   — win-rate + sample + resource tables
- `pa2/runs/dpo_beta/*/summary.json` (if you ran the ablation)